# 09 - Chunk the text and build the Chroma index

We split each document into overlapping chunks and store them in a Chroma collection, so we
can later search them by meaning (semantic search). We use `BAAI/bge-m3` as the embedding
model because it works well for both Finnish and English.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    os.chdir("/content/drive/MyDrive/JobAI")
except ImportError:
    pass

In [ ]:
# !pip install -q chromadb sentence-transformers pyyaml pandas

import re
from pathlib import Path
import yaml
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions

REPO = Path.cwd()
TEXT_DIR = REPO / "data" / "raw" / "rag" / "text"

config = yaml.safe_load(open(REPO / "configs" / "rag.yaml"))
CHUNK_SIZE = config["chunking"]["chunk_size"]
CHUNK_OVERLAP = config["chunking"]["chunk_overlap"]
EMBEDDING_MODEL = config["embedding_candidates"][0]["id"]

documents = pd.read_csv(REPO / "data" / "processed" / "rag" / "documents.csv")
print(len(documents), "documents to chunk")

## Clean the text a little

The PDFs extract with some noise: non-breaking spaces, soft hyphens used as normal hyphens,
and sentences broken across many short lines. We fix the easy parts and leave the rest; a
chunk with a bit of noise in it is still searchable.

In [ ]:
def clean_text(text):
    text = text.replace("\xa0", " ").replace("­", "-")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

## Chunk the text

A simple sliding window over characters: take `chunk_size` characters, move forward by
`chunk_size - chunk_overlap`, repeat. The overlap means a sentence that falls on a chunk
boundary is still fully readable in at least one chunk.

In [ ]:
def chunk_text(text, chunk_size, overlap):
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end].strip()
        if len(chunk) > 50:  # skip tiny leftover fragments
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

In [ ]:
chunk_ids, chunk_texts, chunk_metadatas = [], [], []

def as_date_number(published):
    # Chroma can only compare numbers, not date strings, so we also keep the
    # date as a plain YYYYMMDD number (e.g. "2024-08-31" -> 20240831).
    # A document with no known date (the Statistics Finland releases, since we
    # did not scrape a date for them) gets a number far in the future, so a
    # "before this date" search leaves it out instead of always including it.
    if pd.isna(published):
        return 99999999
    return int(str(published).replace("-", ""))

for row in documents.itertuples():
    text = clean_text((TEXT_DIR / f"{row.doc_id}.txt").read_text(encoding="utf-8"))
    for i, chunk in enumerate(chunk_text(text, CHUNK_SIZE, CHUNK_OVERLAP)):
        chunk_ids.append(f"{row.doc_id}_{i}")
        chunk_texts.append(chunk)
        chunk_metadatas.append({
            "doc_id": row.doc_id,
            "title": str(row.title),
            "language": str(row.language),
            "published": str(row.published),
            "published_number": as_date_number(row.published),
            "source": row.source,
            "url": row.url,
        })

print(len(chunk_texts), "chunks from", documents.doc_id.nunique(), "documents")

## Build the Chroma collection

We use cosine similarity, and let Chroma call the embedding model for us.

In [ ]:
persist_dir = REPO / config["vector_store"]["persist_dir"]
client = chromadb.PersistentClient(path=str(persist_dir))

embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBEDDING_MODEL)

# start clean each time we rebuild the index
existing = [c.name for c in client.list_collections()]
if config["vector_store"]["collection_name"] in existing:
    client.delete_collection(config["vector_store"]["collection_name"])

collection = client.create_collection(
    name=config["vector_store"]["collection_name"],
    embedding_function=embedding_fn,
    configuration={"hnsw": {"space": "cosine"}},
)

In [ ]:
# Chroma is happier with a few hundred documents at a time than all at once
BATCH = 200
for start in range(0, len(chunk_texts), BATCH):
    end = start + BATCH
    collection.add(ids=chunk_ids[start:end], documents=chunk_texts[start:end], metadatas=chunk_metadatas[start:end])

print(collection.count(), "chunks stored in the collection")

## Try a search

Before moving on, check that the collection actually returns something sensible.

In [ ]:
result = collection.query(query_texts=["kuinka monta avointa työpaikkaa"], n_results=3)
for doc, meta, distance in zip(result["documents"][0], result["metadatas"][0], result["distances"][0]):
    print(f"({1 - distance:.2f}) {meta['title']}: {doc[:150]}")